# Mastering Imperfect Information with Deep Recurrent Q-Networks
## Leduc Hold'em — DRQN vs DQN vs Baselines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/DLPW/blob/master/notebooks/DLPW_colab.ipynb)

**This notebook trains and compares 4 agents:**

1. **DRQN** (Deep Recurrent Q-Network with LSTM memory)
2. **DQN** (Standard feedforward DQN - memoryless baseline)
3. **Random Agent** (baseline)
4. **Heuristic Agent** (rule-based baseline)

**Evaluations:**
- DRQN vs Random, Heuristic, DQN
- DQN vs Random, Heuristic
- **Total: 5 matchups**

**Runtime**: GPU recommended (~2-3 hours)

## 1. Setup & Installation

In [ ]:
!pip install -q rlcard torch matplotlib

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Dependencies installed")
print(f"✓ Device: {device}")
print(f"✓ PyTorch version: {torch.__version__}")

## 2. Clone Repository

In [ ]:
import os
import sys

# Clone repository (change YOUR_USERNAME)
!git clone https://github.com/YOUR_USERNAME/DLPW.git
%cd DLPW

sys.path.insert(0, '/content/DLPW')
print("✓ Repository cloned and path configured")

## 3. Configuration

In [ ]:
import config

# Adjust for Colab (increase for better results)
config.NUM_EPISODES_PHASE1 = 10000
config.NUM_EPISODES_PHASE2 = 10000
config.EVALUATE_EVERY = 1000
config.EVALUATE_NUM = 500
config.NUM_EVAL_HANDS = 1000

# Paths
config.OUTPUT_DIR = '/content/DLPW/outputs'
config.MODEL_DIR = '/content/DLPW/outputs/models'
config.PLOT_DIR = '/content/DLPW/outputs/plots'

os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.MODEL_DIR, exist_ok=True)
os.makedirs(config.PLOT_DIR, exist_ok=True)

print("✓ Configuration set")
print(f"  Episodes per phase: {config.NUM_EPISODES_PHASE1}")
print(f"  Expected time: ~2-3 hours on GPU")

## 4. Import Modules

In [ ]:
import rlcard
from rlcard.utils import set_seed
from rlcard.agents import RandomAgent, DQNAgent
import matplotlib.pyplot as plt
import numpy as np
import logging

from config import *
from agents import DRQNAgent, ConservativeHeuristicAgent
from training import CurriculumTrainer
from evaluation import evaluate_agents, compute_action_distribution, print_evaluation_summary
from analysis import plot_training_curves, plot_comparison_curves, plot_ev_comparison_bar

# Suppress RLCard INFO logs
logging.getLogger('rlcard').setLevel(logging.WARNING)

print("✓ All modules imported")

## 5. Environment Setup

In [ ]:
set_seed(SEED)
env = rlcard.make(ENV_NAME)

raw_shape = env.state_shape[0]
state_shape = raw_shape[0] if isinstance(raw_shape, list) else raw_shape
num_actions = env.num_actions

print(f'✓ Environment: {ENV_NAME}')
print(f'✓ State shape: {state_shape}, Actions: {num_actions}')

## 6. Initialize Agents

In [ ]:
# DRQN Agent (with LSTM memory)
agent_drqn = DRQNAgent(
    state_shape=state_shape,
    num_actions=num_actions,
    device=DEVICE,
    hidden_size=HIDDEN_SIZE,
    lr=LEARNING_RATE,
    gamma=GAMMA,
    epsilon_start=EPSILON_START,
    epsilon_min=EPSILON_MIN,
    epsilon_decay=EPSILON_DECAY,
    buffer_capacity=BUFFER_CAPACITY,
    batch_size=BATCH_SIZE,
    min_replay=MIN_REPLAY_SIZE,
    target_update_freq=TARGET_UPDATE_FREQ,
    l2_reg=L2_REGULARIZATION
)

# DQN Agent (feedforward baseline)
agent_dqn = DQNAgent(
    num_actions=num_actions,
    state_shape=env.state_shape[0],
    mlp_layers=[HIDDEN_SIZE, HIDDEN_SIZE],
    device=DEVICE
)

# Baseline agents
agent_random = RandomAgent(num_actions)
agent_heuristic = ConservativeHeuristicAgent(num_actions)

print("✓ All agents initialized")

## 7. Train DRQN (Two-Phase Curriculum)

**Phase 1**: Learn basic card strength against Random agent  
**Phase 2**: Learn exploitation against Heuristic agent

In [ ]:
trainer_drqn = CurriculumTrainer(
    env=env,
    agent=agent_drqn,
    opponent_phase1=agent_random,
    opponent_phase2=agent_heuristic
)

drqn_history = trainer_drqn.train()

# Save model
agent_drqn.save_model(DRQN_CHECKPOINT)
print(f"\n✓ DRQN training complete. Model saved to {DRQN_CHECKPOINT}")

## 8. Train DQN (Baseline Comparison)

In [ ]:
trainer_dqn = CurriculumTrainer(
    env=env,
    agent=agent_dqn,
    opponent_phase1=agent_random,
    opponent_phase2=agent_heuristic
)

dqn_history = trainer_dqn.train()
print("\n✓ DQN training complete")

## 9. Training Curves Visualization

In [ ]:
# DRQN training curves
plot_training_curves(
    ev_history_random=drqn_history['ev_random'],
    ev_history_heuristic=drqn_history['ev_heuristic'],
    loss_history=drqn_history['loss'],
    phase1_episodes=NUM_EPISODES_PHASE1,
    save_path=f'{PLOT_DIR}/drqn_training_curves.png'
)

In [ ]:
# DRQN vs DQN comparison
plot_comparison_curves(
    drqn_hist_random=drqn_history['ev_random'],
    drqn_hist_heuristic=drqn_history['ev_heuristic'],
    dqn_hist_random=dqn_history['ev_random'],
    dqn_hist_heuristic=dqn_history['ev_heuristic'],
    phase1_episodes=NUM_EPISODES_PHASE1,
    save_path=f'{PLOT_DIR}/drqn_vs_dqn_comparison.png'
)

## 10. Final Evaluation

In [ ]:
print(f"\n{'='*60}")
print(f"FINAL EVALUATION ({NUM_EVAL_HANDS} hands per matchup)")
print(f"{'='*60}")

results = {}

# DRQN evaluations
print('\nDRQN vs ...')
ev_vs_random, traj_drqn_vs_random = evaluate_agents(
    env, agent_drqn, agent_random, NUM_EVAL_HANDS, 'Random'
)
ev_vs_heuristic, traj_drqn_vs_heuristic = evaluate_agents(
    env, agent_drqn, agent_heuristic, NUM_EVAL_HANDS, 'Heuristic'
)
ev_vs_dqn, traj_drqn_vs_dqn = evaluate_agents(
    env, agent_drqn, agent_dqn, NUM_EVAL_HANDS, 'Standard DQN'
)

# DQN evaluations
print('\nDQN vs ...')
dqn_vs_random, traj_dqn_vs_random = evaluate_agents(
    env, agent_dqn, agent_random, NUM_EVAL_HANDS, 'Random'
)
dqn_vs_heuristic, traj_dqn_vs_heuristic = evaluate_agents(
    env, agent_dqn, agent_heuristic, NUM_EVAL_HANDS, 'Heuristic'
)

# Store results
results['DRQN vs Random'] = {
    'ev': ev_vs_random,
    'trajectories': traj_drqn_vs_random,
    'action_stats': compute_action_distribution(traj_drqn_vs_random, ACTION_NAMES)
}
results['DRQN vs Heuristic'] = {
    'ev': ev_vs_heuristic,
    'trajectories': traj_drqn_vs_heuristic,
    'action_stats': compute_action_distribution(traj_drqn_vs_heuristic, ACTION_NAMES)
}
results['DRQN vs DQN'] = {
    'ev': ev_vs_dqn,
    'trajectories': traj_drqn_vs_dqn,
    'action_stats': compute_action_distribution(traj_drqn_vs_dqn, ACTION_NAMES)
}
results['DQN vs Random'] = {
    'ev': dqn_vs_random,
    'trajectories': traj_dqn_vs_random,
    'action_stats': compute_action_distribution(traj_dqn_vs_random, ACTION_NAMES)
}
results['DQN vs Heuristic'] = {
    'ev': dqn_vs_heuristic,
    'trajectories': traj_dqn_vs_heuristic,
    'action_stats': compute_action_distribution(traj_dqn_vs_heuristic, ACTION_NAMES)
}

## 11. Evaluation Summary & Analysis

In [ ]:
# Detailed summary with action distributions and bluff rates
print_evaluation_summary(results)

In [ ]:
# EV comparison bar chart
ev_comparison = {
    'DRQN\nvs Random': results['DRQN vs Random']['ev'],
    'DRQN\nvs Heuristic': results['DRQN vs Heuristic']['ev'],
    'DRQN\nvs DQN': results['DRQN vs DQN']['ev'],
    'DQN\nvs Random': results['DQN vs Random']['ev'],
    'DQN\nvs Heuristic': results['DQN vs Heuristic']['ev'],
}
plot_ev_comparison_bar(
    results=ev_comparison,
    save_path=EV_COMPARISON_PLOT
)

## 12. Key Findings

### Success Criterion
**DRQN must outperform DQN** to prove that recurrent memory is beneficial for imperfect information games.

### Expected Results
- **DRQN vs DQN**: Positive EV (memory advantage)
- **Bluff Rate**: 15-20% raising with Jack (proves strategic reasoning)
- **Phase 2 Improvement**: Better performance vs Heuristic after curriculum learning

### Interpretation
- LSTM enables the agent to track betting sequences
- This allows inference of opponent tendencies
- Bluffing behavior emerges without explicit programming

## 13. Download Results

In [ ]:
# Download plots
from google.colab import files

files.download(f'{PLOT_DIR}/drqn_training_curves.png')
files.download(f'{PLOT_DIR}/drqn_vs_dqn_comparison.png')
files.download(f'{PLOT_DIR}/ev_comparison.png')

print("✓ Plots downloaded")

## Summary

### Training Complete
✅ DRQN trained (LSTM memory)  
✅ DQN trained (memoryless baseline)  
✅ 2 baseline agents  
**Total: 4 agents**

### Evaluation Complete
✅ DRQN vs Random, Heuristic, DQN  
✅ DQN vs Random, Heuristic  
**Total: 5 matchups**

### Key Metric
**DRQN vs DQN**: If positive EV, LSTM memory provides advantage!

---

**Training Time**: ~2-3 hours on Colab GPU  
**Episodes**: 10K per phase (increase to 15K+ for better results)